<a href="https://colab.research.google.com/github/mcontreras222/perceptron_Kfold_cross_validation_project/blob/main/mnist_TEMPLATE_train_test_cross_valid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Perceptron with Cross Validation - uses MNIST

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import umap  # pip install umap-learn is the installation command
import numpy as np
from sklearn.datasets import fetch_openml


TEST / TRAIN split. No cross-validation.
I had to be very careful in copying my sample code to change X to X-train; this caused subtle errors where the clustering was clearly not working right but the code was running

In [2]:

# mnist as a numpy array
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
X = X[:50000] / 255.0  # Normalize pixels
y = y[:50000]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)

# reduce dimensions with  umap - this takes 47 seconds
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1)
X_red_train = reducer.fit_transform(X_train)
X_red_test = reducer.transform(X_test)
from sklearn.linear_model import Perceptron
ppn = Perceptron(eta0=0.1, random_state=1)
ppn.fit(X_red_train, y_train) # train the algorithm


Perceptron(eta0=0.1, random_state=1)

In [3]:
print(y_test[:10])  # print the first 10 test labels. They are strings, not numbers.

['2' '2' '5' '5' '9' '7' '2' '7' '3' '4']


In [4]:
# use the test data
y_pred = ppn.predict(X_red_test)
print('Misclassified examples: %d' % (y_test != y_pred).sum())

Misclassified examples: 8571


Define your own accuracy function.  

In [8]:
def calculate_accuracy(y, y_pred):
  c = 0
  for i in range(len(y)):
    if y[i]==y_pred[i]: c+=1
  return c/len(y)

test  with toy example

In [9]:
def test_accuracy():
    y_pred = [1,0,0,0]
    y = [0,0,0,1]
    accuracy = calculate_accuracy(y, y_pred)
    print(f"Should be 0.50: {accuracy:.4f}")
test_accuracy()

Should be 0.50: 0.5000


In [10]:
calculate_accuracy(y_test, y_pred) # call your function

0.57145

Here's the 'real' way to do it:

In [12]:
print('Train set accuracy: %.3f' % ppn.score(X_red_train, y_train))  #about 52% accurate
# do the same for the test set - YOUR CODE HERE
print('Test set accuracy: %.3f' % ppn.score(X_red_test, y_test))

Train set accuracy: 0.573
Test set accuracy: 0.571


# now try cross-validation

In [31]:
from sklearn.model_selection import StratifiedKFold
kfold = StratifiedKFold(n_splits=10).split(X_train, y_train)
scores = []
for k, (train, test) in enumerate(kfold):
   ppn.fit(X_train[train], y_train[train])
   score = ppn.score(X_train[test], y_train[test])
   scores.append(score)
   print(f'Fold: {k+1:02d}, 'f'Acc.: {score:.3f}')

Fold: 01, Acc.: 0.890
Fold: 02, Acc.: 0.854
Fold: 03, Acc.: 0.868
Fold: 04, Acc.: 0.882
Fold: 05, Acc.: 0.876
Fold: 06, Acc.: 0.865
Fold: 07, Acc.: 0.821
Fold: 08, Acc.: 0.878
Fold: 09, Acc.: 0.896
Fold: 10, Acc.: 0.840


In [32]:
mean_acc = np.mean(scores)
std_acc = np.std(scores)
print(f'\n Cross Validation accuracy: {mean_acc:.3f} +/- {std_acc:.3f}')


 Cross Validation accuracy: 0.867 +/- 0.022


This is fine, but if we want to try changing a parameter (like the number of folds), then we need to do train/validation/test split not just test/train split.
Since we didn't actually use our entire MNIST dataset, we have effectively 'held out' some data and the test & train data is separate. So, we could optimize a parameter based on the method above and then when we are all done, get the final accuracy of the best choice by testing the trained perceptron on the unused data.

Automate (using a loop) the calculation of the (mean) CV accuracy - loop over choices of K, which is the number of folds. Select the best value of K and then get a final accuracy value using the unused data.

In [39]:
from sklearn.model_selection import StratifiedKFold
best_acc = -1
best_k = -1
best_mean = -1
best_std = -1
for i in range(7,11):
  kfold = StratifiedKFold(n_splits=i).split(X_train, y_train)
  scores = []
  for k, (train, test) in enumerate(kfold):
    ppn.fit(X_train[train], y_train[train])
    score = ppn.score(X_train[test], y_train[test])
    scores.append(score)
    # print(f'Fold: {k+1:02d}, 'f'Acc.: {score:.3f}')
  mean_acc = np.mean(scores)
  std_acc = np.std(scores)
  acc = mean_acc - std_acc
  if acc > best_acc:
    best_acc = acc
    best_mean = mean_acc
    best_std = std_acc
    best_k = i

In [40]:
print(f'Best k: {best_k}\nCross Validation Accuracy: {best_mean:.3f} +/- {best_std:.3f}')


Best k: 9
Cross Validation Accuracy: 0.871 +/- 0.009


With Test and k splits = 9 because that gave the best accuracy in the previous blocks.

In [43]:
kfold = StratifiedKFold(n_splits=9).split(X_test, y_test)
scores = []
for k, (train, test) in enumerate(kfold):
   ppn.fit(X_train[train], y_train[train])
   score = ppn.score(X_train[test], y_train[test])
   scores.append(score)
   print(f'Fold: {k+1:02d}, 'f'Acc.: {score:.3f}')

mean_acc = np.mean(scores)
std_acc = np.std(scores)
print(f'\n Cross Validation accuracy: {mean_acc:.3f} +/- {std_acc:.3f}')

Fold: 01, Acc.: 0.864
Fold: 02, Acc.: 0.867
Fold: 03, Acc.: 0.864
Fold: 04, Acc.: 0.839
Fold: 05, Acc.: 0.853
Fold: 06, Acc.: 0.827
Fold: 07, Acc.: 0.882
Fold: 08, Acc.: 0.867
Fold: 09, Acc.: 0.886

 Cross Validation accuracy: 0.861 +/- 0.018
